# Lab 3: Predicting Prices and Quantities

> Requires the artifact written by **Lab 2**.

Before estimating a causal effect it helps to know what the controls can predict. In the
partially linear model of Lab 4,

$$Q_{it}^{\perp} = \delta\, P_{it}^{\perp} + e_{it}, \qquad
Q^{\perp}_{it} = Q_{it} - E[Q_{it}\mid S_{it}], \quad
P^{\perp}_{it} = P_{it} - E[P_{it}\mid S_{it}],$$

everything rests on the two nuisance functions $E[Q\mid S]$ and $E[P\mid S]$. This
notebook measures how well we can estimate them, and which kind of variation the
fine-tuned embeddings explain.

## Four targets

We evaluate levels and changes:

| Target | Meaning |
|---|---|
| $Q_{it}$ | level of the quantity signal |
| $P_{it}$ | level of the price signal |
| $\Delta Q_{it}$ | period-to-period change in quantity |
| $\Delta P_{it}$ | period-to-period change in price |

A confounder of the price-quantity relationship has to explain changes, since it must move
price and quantity together over time. A variable that explains only levels, that is, why
one product is generally pricier and better selling than another, is largely absorbed by
the product's own history and does not bias the elasticity much. It can still tell us for
which products the elasticity is large or small.

The gap between levels and changes therefore decides whether the embeddings act as
confounders, which is Lab 4's concern, or as effect modifiers, which is Lab 5's.

Models are fitted on the fine-tune products and scored on the estimation products, which
are entirely held out. Following the paper, negative out-of-sample $R^2$ is reported as 0.

## Setup

In [ ]:
%pip install -q lightgbm

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

palette = sns.color_palette("colorblind")
pd.set_option("display.width", 140)

## Loading the artifact from Lab 2

In [ ]:
ARTIFACT = "subsample_v2.parquet"
drive_path = f"/content/drive/MyDrive/demand_labs_v2/{ARTIFACT}"

# Colab gives every notebook its own machine, so /content does not carry across
# labs. Google Drive does.
if os.path.isdir("/content") and not os.path.exists(drive_path):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"could not mount Drive ({type(exc).__name__})")

path = next((p for p in [drive_path, f"/content/{ARTIFACT}", ARTIFACT]
             if os.path.exists(p)), None)
if path is None:
    raise FileNotFoundError(
        "subsample_v2.parquet not found. Run Lab 2 "
        "(02_finetune_embeddings.ipynb) first and let it save to Google Drive, "
        "or upload the file into this session.")

df = pd.read_parquet(path)
print(f"loaded {path}")
print(f"{df['ASIN'].nunique():,} products, {len(df):,} rows, "
      f"{df['period'].nunique()} periods")
print(df.groupby("split").agg(products=("ASIN", "nunique"), rows=("ASIN", "size")))

## Feature sets

Four nested specifications, following Tables 5 and 6 of the paper:

* **Tabular**: human-encoded features only, meaning subcategory and period dummies plus
  the time-varying controls. No embeddings.
* **+ 5 PCA**: adds the first five principal components of $X^e$.
* **+ 5 Similarities**: adds the five cosine similarities to the $k$-means centroids.
* **+ 256 Embeddings**: adds the full projected embedding.

All four include the time-varying tabular controls (lagged rating and review count, offer
counts, deal and FBA flags). Each is then run twice, once without lagged $Q$ and $P$,
which is the comparison the paper's Tables 5 and 6 report, and once with them, which is
the full state $S_{it}$ that Lab 4 conditions on. The contrast between the two blocks is
what the notebook is for.

In [ ]:
emb_cols = [c for c in df.columns if c.startswith("emb_")]
pca_cols = [c for c in df.columns if c.startswith("pca_")]
sim_cols = [c for c in df.columns if c.startswith("similarity_cluster_")]
sub_cols = [c for c in df.columns if c.startswith("sub_")]

lag_controls = ["Q_t-1", "P_bb_t-1"]
tab_controls = ["RATING_t-1", "REVIEW_COUNT_t-1", "n_offers",
                "n_offers_fba", "n_offers_fbm", "lightning_deal", "is_fba"]

periods = sorted(df["period"].unique())[1:]   # first period is the baseline
period_cols = [f"period_{p}" for p in periods]
for p in periods:
    df[f"period_{p}"] = (df["period"] == p).astype(int)

BASE = sub_cols + period_cols + tab_controls
FEATURE_SETS = {
    "Tabular":          BASE,
    "+ 5 PCA":          BASE + pca_cols,
    "+ 5 Similarities": BASE + sim_cols,
    "+ 256 Embeddings": BASE + emb_cols,
}
STATE = lag_controls + tab_controls   # dropna basis: same rows for every spec
TARGETS = ["Q_t", "P_bb_t", "Delta_Q_t", "Delta_P_bb_t"]

print(f"{len(emb_cols)} embeddings | {len(pca_cols)} PCA | {len(sim_cols)} "
      f"similarities | {len(sub_cols)} subcategories | {len(period_cols)} periods")

## Fitting

For every combination of feature set, lag block and target we fit a linear model and
gradient boosted trees. Rows whose lag is undefined, that is the first period of each
product, are dropped throughout, so every specification is scored on the same
observations.

In [ ]:
data = df.dropna(subset=STATE + TARGETS).copy()
train = data[data["split"] == "finetune"]
test = data[data["split"] == "estimation"]
print(f"train: {len(train):,} rows ({train['ASIN'].nunique()} products)   "
      f"test: {len(test):,} rows ({test['ASIN'].nunique()} products)")


def score(features, target):
    """Out-of-sample R^2 for OLS and LightGBM; negatives clipped to 0."""
    X_tr, X_te = train[features], test[features]
    y_tr, y_te = train[target], test[target]

    ols = sm.OLS(y_tr, sm.add_constant(X_tr, has_constant="add")).fit()
    r2_ols = r2_score(y_te, ols.predict(sm.add_constant(X_te, has_constant="add")))

    gbm = LGBMRegressor(n_estimators=500, learning_rate=0.02,
                        random_state=42, verbose=-1).fit(X_tr, y_tr)
    r2_gbm = r2_score(y_te, gbm.predict(X_te))

    return {"OLS": max(r2_ols, 0.0), "LightGBM": max(r2_gbm, 0.0)}


rows = []
for with_lags in (False, True):
    label = "with lagged Q,P" if with_lags else "no lagged Q,P"
    for name, feats in FEATURE_SETS.items():
        cols = feats + (lag_controls if with_lags else [])
        for target in TARGETS:
            for model, value in score(cols, target).items():
                rows.append({"Controls": label, "Features": name, "Model": model,
                             "Target": target, "R2": value})
        print(f"  {label:13s} | {name:18s} done", flush=True)

results = pd.DataFrame(rows)

## Results

In [ ]:
table = (results
         .pivot_table(index=["Controls", "Features", "Model"], columns="Target",
                      values="R2")
         .reindex(columns=TARGETS)
         .reindex(["no lagged Q,P", "with lagged Q,P"], level="Controls")
         .reindex(list(FEATURE_SETS), level="Features"))
print("Out-of-sample R^2 (%)")
(table * 100).round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for ax, state in zip(axes, ["no lagged Q,P", "with lagged Q,P"]):
    sub = results[(results["Controls"] == state) & (results["Model"] == "LightGBM")]
    wide = sub.pivot(index="Features", columns="Target",
                     values="R2").reindex(list(FEATURE_SETS))
    x = np.arange(len(wide))
    for i, t in enumerate(TARGETS):
        ax.bar(x + (i - 1.5) * 0.2, wide[t] * 100, 0.2, label=t, color=palette[i])
    ax.set_xticks(x)
    ax.set_xticklabels(wide.index, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("out-of-sample $R^2$ (%)")
    ax.set_title(f"LightGBM, {state}")
    ax.grid(axis="y", alpha=0.3)
axes[0].legend(fontsize=9)
plt.tight_layout()
plt.show()

### Levels versus changes

To see the asymmetry directly, compare what the embeddings add on top of the tabular
baseline for each target.

In [ ]:
gain = (results[results["Model"] == "LightGBM"]
        .pivot_table(index=["Controls", "Features"], columns="Target", values="R2"))
delta = (gain.xs("+ 256 Embeddings", level="Features")
         - gain.xs("Tabular", level="Features"))[TARGETS]

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(TARGETS))
for i, state in enumerate(["no lagged Q,P", "with lagged Q,P"]):
    ax.bar(x + (i - 0.5) * 0.35, delta.loc[state] * 100, 0.35,
           label=state, color=palette[i])
ax.axhline(0, color="gray", lw=1)
ax.set_xticks(x)
ax.set_xticklabels(TARGETS)
ax.set_ylabel("percentage points of $R^2$")
ax.set_title("What the 256 embeddings add over the tabular baseline (LightGBM)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print((delta * 100).round(2).to_string())

## Reading the table

**Boosted trees beat OLS, and embeddings beat hand-coded tabular features.** The relation
between what a product is and what it costs is not linear, and a subcategory label is a
coarse summary of a product.

**Five numbers do nearly as much work as 256.** The principal components and the cluster
similarities recover most of what the full embedding contributes. This is what lets Lab 5
use five similarities as the variables the elasticity depends on: the heterogeneity model
stays small enough to interpret without discarding much information.

**Changes are much harder to predict than levels.** In the paper's full sample the best
models reach roughly 50 to 60% $R^2$ for $Q_{it}$ and 65% for $P_{it}$, but only about 15%
for $\Delta Q_{it}$ and 1.5% for $\Delta P_{it}$. The direction of the gap is what
matters.

**Once the product's own history is in the model, the embeddings stop mattering.** Compare
the two blocks. Without lagged $Q$ and $P$ the embeddings add tens of points of $R^2$ on
the levels. With them they add almost nothing, because $Q_{i,t-1}$ already encodes how
popular and how visible a product is, which is most of what the description and image were
telling us. This is the paper's finding that lagged quantity is the key confounder, and it
is why Lab 4 builds its state around it.

### Consequences for the next two labs

A variable confounds the price-quantity relationship only if it moves both. The embeddings
barely predict $\Delta P$: knowing what a product is tells you little about when its price
will move. Adding them as controls should therefore leave the estimated elasticity roughly
unchanged, which is what Lab 4 finds.

What the embeddings do predict is levels, meaning which products are expensive and which
sell well. Those are plausible determinants of how sensitive a product is to a price cut,
so Lab 5 lets the elasticity be a function of them.

> **Notes on your numbers.**
>
> * The estimation sample here is roughly a sixth of the paper's, so these $R^2$ values
>   are noisier than the published ones and adjacent rows may swap order. The gap between
>   levels and changes is large enough to survive.
> * OLS on the 256-dimensional embedding is fitted on only about 300 products and will
>   often overfit badly enough to score a negative out-of-sample $R^2$, reported here as
>   0. That is a small-sample artifact rather than a finding, so read the boosted-tree
>   row. It goes away with more products.